# 🧪 Thực Nghiệm Toàn Diện: Bộ 5 Bài Test Khoa Học Chứng Minh Điểm Yếu Của LiDAR
### Khung Lý Thuyết: Dimension-Free Lipschitz Bound, Particle Filtering (SMC) & Randomized Smoothing (RS-LiDAR)

Notebook này thiết lập môi trường thực nghiệm hoàn chỉnh để kiểm chứng toàn bộ **Bộ 5 Bài Test Khoa Học Độc Lập** so sánh giữa phương pháp **LiDAR gốc (ICML 2026)** và **Phương pháp Của Bạn (Smoothed Surrogate / RS-LiDAR)**:

---

### 📌 Hệ Thống 5 Bài Test Khoa Học:
1. **🧪 TEST 1: Kháng Sai số Bộ giải (Solver Error Robustness & Theorem 1)**
   * *Lý thuyết*: Khi dùng DPM-Solver 5 bước, sai số hình học $\mathbf{e}_i = \hat{\mathbf{x}}_0^i - \mathbf{x}_0^i$ khiến hàm thưởng thô $r(\hat{\mathbf{x}}_0)$ bùng nổ do $L_0 \to \infty$. RS-LiDAR chặn trên sai số bằng hằng số Lipschitz hữu hạn: $|r_\sigma(\hat{\mathbf{x}}_0) - r_\sigma(\mathbf{x}_0)| \le L_\sigma \|\mathbf{e}_i\|_2$.
   * *Chỉ số đo*: Sai số điểm thưởng $|\Delta r|$ và hệ số tương quan thứ bậc Kendall $\tau$ giữa 5 bước DPM và 50 bước DDIM chuẩn trên đa mô hình reward (ImageReward, CLIP-Score, HPS v2.1).

2. **🧪 TEST 2: Kháng Sụp đổ Trọng số Softmax (Softmax Mode Collapse Prevention)**
   * *Lý thuyết*: Hàm $\exp(\lambda r)$ của LiDAR với $\lambda=5000$ bị bão hòa One-Hot tại các đỉnh gai nhọn cục bộ, dồn $95\%$ trọng số vào đúng 1 hạt (Best-of-1 Trap). RS-LiDAR làm phẳng các đỉnh nhọn, duy trì phân bổ trọng số đa hạt mượt mà.
   * *Chỉ số đo*: Entropy Shannon $H(w^r) = -\sum w_i^r \log_2 w_i^r$ xuyên suốt 50 bước khử nhiễu $t \in [1000, 0]$.

3. **🧪 TEST 3: Kháng Rung lắc Vector Dẫn đường (Guidance Field Lipschitz Stability)**
   * *Lý thuyết*: Vector dẫn đường $\mathbf{g}_t(\mathbf{x}_t)$ của LiDAR bị bẻ ngoặt hỗn loạn khi trạng thái $\mathbf{x}_t$ dao động nhỏ $\delta$. RS-LiDAR đảm bảo độ nhạy ma trận Jacobi $\|\frac{\partial \mathbf{g}_t}{\partial \mathbf{x}_t}\| \le C \cdot L_\sigma < \infty$.
   * *Chỉ số đo*: Độ ổn định góc quay $\text{CosSim}(\mathbf{g}_t, \mathbf{g}_{t+\delta})$ tại nhiễu vi mô $\|\delta\|_2 = 10^{-3}$.

4. **🧪 TEST 4: Đo Mức Độ Suy Thoái Hạt Hữu Hiệu (Effective Sample Size - ESS & Particle Starvation)**
   * *Lý thuyết (Chuẩn mực Sequential Monte Carlo / Particle Filter)*: Đo số hạt hữu hiệu $\text{ESS}_t = \frac{1}{\sum (w_i^r)^2}$. LiDAR bị sụp đổ $\text{ESS} \approx 1.05 - 1.20$, bóc trần sự thật rằng 98% chi phí tính toán của 50 hạt bị lãng phí. RS-LiDAR duy trì $\text{ESS} \ge 15 - 30$, kích hoạt sức mạnh đa hạt thực thụ.
   * *Chỉ số đo*: $\text{ESS}_t$, Normalized ESS ($\text{NESS}$), trọng số lớn nhất $w_{\max}$ và số hạt tích cực $N_{\text{active}}$.

5. **🧪 TEST 5: Khảo Sát Giới Hạn Bước Bộ Giải Nhanh (Step-Budget Solver Scaling: $S \in \{2, 3, 5, 8, 15\}$)**
   * *Lý thuyết*: LiDAR chọn $S=5$ theo cảm tính. Khi giảm số bước $S < 5$ để tăng tốc suy luận, sai số bộ giải $\|\mathbf{e}_S\|_2$ tăng vọt khiến LiDAR sụp đổ thứ bậc $\tau$ thẳng đứng. Nhờ chặn Lipschitz Định lý 1, RS-LiDAR tại **$S=3$ bước** vẫn đạt độ chính xác tương đương hoặc vượt trội LiDAR gốc ở $S=5$ bước $\implies$ Giúp tăng tốc độ sinh ảnh gấp gần 2 lần!
   * *Chỉ số đo*: Đường cong thoái hóa bước $\tau(S)$ và sai số $|\Delta r(S)|$ so với chuẩn 50 bước DDIM.

## 1. Kiểm Tra Phần Cứng GPU (T4 / L4 / A100)

In [ ]:
import torch, sys
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)} GB")
else:
    raise RuntimeError("❌ Vui lòng chọn Runtime -> Change runtime type -> GPU!")
!nvidia-smi

## 2. Gắn Kết Google Drive & Đồng Bộ Mã Nguồn Repo

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    base_drive = "/content/drive/My Drive" if os.path.exists("/content/drive/My Drive") else "/content/drive/MyDrive"
    DRIVE_DIR = f"{base_drive}/RS-LiDAR/test_results_5_tests"
except Exception:
    DRIVE_DIR = "experiments/test_results_5_tests"

os.makedirs(DRIVE_DIR, exist_ok=True)

REPO_DIR = "/content/RS-LiDAR"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/leekwanreal/RS-LiDAR.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin main

WORKDIR = REPO_DIR if os.path.exists(f"{REPO_DIR}/test_lidar_weaknesses.py") else f"{REPO_DIR}/Diffusion-LiDAR-Sampling"
%cd {WORKDIR}
print("✅ Thư mục làm việc hiện tại:", os.getcwd())
print("📁 Thư mục lưu kết quả:", DRIVE_DIR)

## 3. Cài Đặt Môi Trường Đầy Đủ (Transformers, Diffusers, ImageReward, CLIP, HPSv2)

In [ ]:
import os, urllib.request

os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q --upgrade protobuf
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q git+https://github.com/THUDM/ImageReward.git
!pip install -q hpsv2 matplotlib tqdm scipy seaborn pandas tabulate

# Vá lỗi từ điển hpsv2 nếu thiếu
try:
    import hpsv2
    hpsv2_vocab = os.path.join(os.path.dirname(hpsv2.__file__), "src", "open_clip", "bpe_simple_vocab_16e6.txt.gz")
    os.makedirs(os.path.dirname(hpsv2_vocab), exist_ok=True)
    if not os.path.exists(hpsv2_vocab):
        urllib.request.urlretrieve("https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz", hpsv2_vocab)
except Exception as e:
    print(f"Chú ý hpsv2: {e}")

print("✅ Môi trường cho Bộ 5 Bài Test đã sẵn sàng 100%!")

## 4. [LỰA CHỌN NHANH] Chạy Riêng Bài Test 4 (Đo ESS & Sụp Đổ Hạt)
*Thời gian chạy*: **Chỉ mất khoảng 15 - 20 giây!**
Bài test này đo Effective Sample Size (ESS), Normalized ESS, và $w_{\max}$ theo chuẩn Sequential Monte Carlo để chứng minh hiện tượng Best-of-1 Trap của LiDAR.

In [ ]:
!python test_lidar_weaknesses.py \
    --test 4 \
    --num_particles 50 \
    --sigma 0.25 \
    --tune_sigma \
    --sigmas "0.10,0.25,0.50,1.00" \
    --output_dir "{DRIVE_DIR}"

## 5. [LỰA CHỌN NHANH] Chạy Riêng Bài Test 5 (Khảo Sát Bước Bộ Giải $S \in \{2, 3, 5, 8, 15\}$)
*Thời gian chạy*: ~5 - 10 phút (với 10 prompts, 10 particles).
Bài test này kiểm chứng Định lý 1 Dimension-Free Lipschitz Bound, chứng minh RS-LiDAR tại $S=3$ đạt độ chính xác ngang ngửa LiDAR tại $S=5$.

In [ ]:
!python test_lidar_weaknesses.py \
    --test 5 \
    --num_prompts 10 \
    --num_particles 10 \
    --sigma 0.05 \
    --output_dir "{DRIVE_DIR}"

## 6. [TOÀN DIỆN] Chạy Toàn Bộ Hệ Thống 5 Bài Test Khoa Học (`--test all`)
Chạy trọn gói từ Test 1 đến Test 5, tự động xuất bảng CSV/Markdown và biểu đồ tổng hợp 5 bài test ra Google Drive.

In [ ]:
!python test_lidar_weaknesses.py \
    --test all \
    --num_prompts 20 \
    --num_particles 20 \
    --sigma 0.25 \
    --tune_sigma \
    --sigmas "0.10,0.25,0.50,1.00" \
    --output_dir "{DRIVE_DIR}"

## 7. Hiển Thị Bảng Kết Quả Khoa Học & Toàn Bộ Đồ Thị So Sánh Trực Quan

In [ ]:
import pandas as pd, glob, os
from IPython.display import display, Image, Markdown

# 1. Hiển thị Bảng so sánh khoa học các bài test
table_files = glob.glob(f"{DRIVE_DIR}/*comparison*.csv")
if table_files:
    df = pd.read_csv(table_files[0])
    print("📊 BẢNG TỔNG HỢP KẾT QUẢ CÁC BÀI TEST:")
    display(df)
    
    md_files = glob.glob(f"{DRIVE_DIR}/*comparison*.md")
    if md_files:
        with open(md_files[0], 'r', encoding='utf-8') as f:
            display(Markdown(f.read()))

# 2. Hiển thị Bảng khảo sát Sigma Ablation
sigma_files = glob.glob(f"{DRIVE_DIR}/*sigma*.csv")
if sigma_files:
    df_sigma = pd.read_csv(sigma_files[0])
    print("\n📈 BẢNG KHẢO SÁT THAM SỐ BÁN KÍNH LÀM MỊN SIGMA:")
    display(df_sigma)

# 3. Hiển thị tất cả các biểu đồ khoa học đã xuất
png_files = sorted(glob.glob(f"{DRIVE_DIR}/*.png"))
for p in png_files:
    print(f"\n🖼️ Đồ thị: {os.path.basename(p)}")
    display(Image(filename=p))